# Averaged Point Metrics Reward Mode Collapse

Probabilistic suffix prediction models forecast a distribution over potential future suffixes. 
Traditionally, these models are evaluated by drawing $K$ suffixes $\hat{s}^{(1)}, \dots, \hat{s}^{(K)}$, scoring
each against the observed suffix $s$ with a point metric $m$ (such as the Damerau-Levenshtein
similarity (DLS), absolute length error or remaining time error), and averaging:

$$
\bar{m} = \frac{1}{K} \sum_{l=1}^{K} m\bigl(\hat{s}^{(l)}, s\bigr).
$$

For a fixed prefix, the expected score is the average of $g(\hat{s}) = \mathbb{E}_s\,[m(\hat{s}, s)]$
across the draws. Because an average can never exceed its highest term, a deterministic model that
repeatedly predicts the single suffix maximizing $g(\hat{s})$ will score at least as well as any model that
faithfully captures the true distribution. Consequently, averaged point metrics are **improper scoring rules**:
they structurally reward mode collapse.

In this notebook, we demonstrate this flaw empirically using a simulated process with a known true distribution. 
We compare an oracle (which draws $K$ suffixes from the true distribution) against a mode model 
(which always predicts the most probable suffix). If the averaged metrics score the mode model at least as well as the oracle, 
it confirms they fail to reward predicting the true distribution.

In [1]:
import numpy as np
import pandas as pd

from improper_suffix_metrics.evaluation import evaluate, predict
from improper_suffix_metrics.markov_process import (
    END,
    Activity,
    LogNormal,
    MarkovProcess,
    Trace,
    randomize,
)

# Random seed for reproducibility
SEED = 42
# Number of cases to simulate for the test set
TEST_CASE_COUNT = 1000
# Number of draws for each prefix
PREDICTED_DRAW_COUNT = 20

In [2]:
def to_frame(traces: list[Trace]) -> pd.DataFrame:
    """Show each trace as a row with its activities, length, and total time in days."""
    return pd.DataFrame(
        {
            'activities': [str(trace) for trace in traces],
            'length': [len(trace) for trace in traces],
            'time (days)': [round(trace.duration, 2) for trace in traces],
        }
    )

## The process

To demonstrate empirically that averaged point metrics reward mode collapse, we will use a simple loan application process. The process begins with the submission of a loan application, after which it diverges into two branches:
- **Fast track ($p$)**: Handled via an automated check and immediate approval;
- **Manual review ($1-p$)**: requires additional steps, including documentation requests, manual evaluation and approval.

The duration of each activity is modeled as a random variable following a **log-normal distribution**, parametrized by activity-specific values of $\mu$ and $\sigma$. The complete process structure is illustrated below:

![Loan application process](figures/loan_process.svg)

### Scenario 1: High automation setting

We will first examine a scenario where the automated fast track is significantly more frequent than manual review. Specifically, we set the fast-track probability to $p = 0.7$.

In [3]:
process = MarkovProcess(
    start={'Submit application': 1.0},
    activities={
        'Submit application': Activity(
            time=LogNormal(median=0.1, spread=0.5),
            successors={'Automatic check': 0.7, 'Request documents': 0.3},
        ),
        'Automatic check': Activity(
            time=LogNormal(median=1.0 / 24.0, spread=0.5),
            successors={'Approve': 1.0},
        ),
        'Approve': Activity(
            time=LogNormal(median=0.5, spread=0.5),
            successors={END: 1.0},
        ),
        'Request documents': Activity(
            time=LogNormal(median=2.0, spread=0.5),
            successors={'Manual assessment': 1.0},
        ),
        'Manual assessment': Activity(
            time=LogNormal(median=4.0, spread=0.5),
            successors={'Final decision': 1.0},
        ),
        'Final decision': Activity(
            time=LogNormal(median=1.0, spread=0.5),
            successors={END: 1.0},
        ),
    },
)

### Test Case Generation

Using this process model, we simulate a set of test cases. Each generated case follows one of the two defined pathways, complete with realistic timestamps sampled according to the respective activity distributions.

In [4]:
rng = np.random.default_rng(SEED)
cases = [process.sample(rng) for _ in range(TEST_CASE_COUNT)]
to_frame(cases[:10])

,activities,length,time (days)
0,Submit application -> Request documents -> Man...,4,6.14
1,Submit application -> Request documents -> Man...,4,8.32
2,Submit application -> Request documents -> Man...,4,5.96
3,Submit application -> Automatic check -> Approve,3,0.51
4,Submit application -> Automatic check -> Approve,3,0.72
5,Submit application -> Automatic check -> Approve,3,0.56
6,Submit application -> Automatic check -> Approve,3,0.73
7,Submit application -> Automatic check -> Approve,3,0.36
8,Submit application -> Automatic check -> Approve,3,0.80
9,Submit application -> Automatic check -> Approve,3,0.34


As expected, the proportion of cases taking each route closely reflects our probability parameter p.

In [5]:
pd.Series([str(case) for case in cases], name='activities').value_counts(normalize=True).round(3)

activities
Submit application -> Automatic check -> Approve                                  0.694
Submit application -> Request documents -> Manual assessment -> Final decision    0.306
Name: proportion, dtype: float64

### The two forecasters

Within our process model framework, we evaluate and compare the performance of two distinct suffix prediction strategies:
- **The Oracle**: Draws `PREDICTED_DRAW_COUNT` suffixes directly from the true underlying distribution, serving as a baseline for optimal performance.
- **The Mode Model**: Predicts a single deterministic suffix corresponding to the most probable path, with each activity scheduled at its median duration.


In [6]:
def oracle(prefix: Trace) -> list[Trace]:
    return [process.sample_suffix(prefix, rng) for _ in range(PREDICTED_DRAW_COUNT)]

# Doesn't need to draw multiple suffixes, since the mode is deterministic.
# We can simply return the mode suffix as a single-element list.
def mode(prefix: Trace) -> list[Trace]:
    return [process.mode_suffix(prefix)]

### Predictions

Each forecaster predicts every prefix of every test case that leaves a non-empty suffix.

In [7]:
# Predict the suffixes for each test case using the oracle
oracle_predictions = predict(oracle, cases)

# Predict the suffixes for each test case using the mode model
mode_predictions = predict(mode, cases)

The first prediction shows how the two forecasters differ. After *Submit application*, the
oracle's draws split between the two tracks, while the mode model puts its single draw on the
fast track at its median times.

In [8]:
pd.concat(
    {
        'observed': to_frame([oracle_predictions[0].observed]),
        'mode': to_frame(mode_predictions[0].draws),
        'oracle': to_frame(oracle_predictions[0].draws),
    }
)

activities  length  \
observed 0   Request documents -> Manual assessment -> Fina...       3   
mode     0                          Automatic check -> Approve       2   
oracle   0   Request documents -> Manual assessment -> Fina...       3   
         1                          Automatic check -> Approve       2   
         2                          Automatic check -> Approve       2   
         3                          Automatic check -> Approve       2   
         4                          Automatic check -> Approve       2   
         5                          Automatic check -> Approve       2   
         6                          Automatic check -> Approve       2   
         7   Request documents -> Manual assessment -> Fina...       3   
         8                          Automatic check -> Approve       2   
         9                          Automatic check -> Approve       2   
         10                         Automatic check -> Approve       2   
         11                         Automatic check -> Approve       2   
         12                         Automatic check -> Approve       2   
         13  Request documents -> Manual assessment -> Fina...       3   
         14                         Automatic check -> Approve       2   
         15                         Automatic check -> Approve       2   
         16                         Automatic check -> Approve       2   
         17                         Automatic check -> Approve       2   
         18  Request documents -> Manual assessment -> Fina...       3   
         19                         Automatic check -> Approve       2   

             time (days)  
observed 0          6.14  
mode     0          0.54  
oracle   0          5.43  
         1          0.50  
         2          0.36  
         3          0.56  
         4          0.68  
         5          0.68  
         6          0.71  
         7         11.39  
         8          0.33  
         9          1.14  
         10         0.63  
         11         0.60  
         12         0.65  
         13         4.75  
         14         0.39  
         15         0.53  
         16         0.77  
         17         0.34  
         18         6.46  
         19         0.82

### Comparison

Both forecasters are evaluated on each prefix with the following metrics, which compare the $K$ drawn suffixes $\hat{s}^{(l)}$ with the observed suffix $s$ and are then averaged over all prefixes:

-   **Damerau-Levenshtein Similarity** (higher is better), on the activity sequences $\hat{a}^{(l)} = \operatorname{act}(\hat{s}^{(l)})$ and $a = \operatorname{act}(s)$:

    $$
    \operatorname{DLS} = \frac{1}{K} \sum_{l=1}^{K} \left( 1 - \frac{d_{\mathrm{DL}}\bigl(\hat{a}^{(l)}, a\bigr)}{\max\bigl(|\hat{a}^{(l)}|, |a|\bigr)} \right),
    $$

    where $d_{\mathrm{DL}}$ is the minimum number of insertions, deletions, substitutions, and adjacent transpositions turning $\hat{a}^{(l)}$ into $a$.

-   **Mean Absolute Error of the Suffix Length** (lower is better), between the numbers of activities $|\hat{s}^{(l)}|$ and $|s|$.

-   **Mean Absolute Error of the Remaining Time** (lower is better), between the remaining times $\hat{R}^{(l)}$ and $R$ in days.

In [9]:
pd.DataFrame(
    {
        'Oracle': evaluate(oracle_predictions),
        'Mode': evaluate(mode_predictions),
    }
).T.round(3)

,DLS,Length MAE,Remaining Time MAE
Oracle,0.815,0.185,2.042
Mode,0.867,0.133,1.375


It can be clearly seen that the mode model outperforms the oracle on all three metrics, confirming that averaged point metrics reward mode collapse and fail to incentivize accurate distributional predictions.

## Random settings

To ensure that the results are not specific to the chosen process parameters, the experiment is repeated across a range of random settings. 
Each setting preserves the process structure while randomly drawing: 
- $p \sim U(0, 1)$ for the branching probability,
- The log-normal duration parameters for each activity, $\mu \sim U(\log(1h), \log(10d))$ and $\sigma \sim U(0.1, 1)$.

For each setting, the pipeline simulates test cases, runs both forecasters on every prefix, and computes their scores.

In [10]:
SETTING_COUNT = 500


def compare(process: MarkovProcess, rng: np.random.Generator) -> dict[str, dict[str, float]]:
    """Simulate test cases from the process and evaluate the oracle and the mode model on them."""
    cases = [process.sample(rng) for _ in range(TEST_CASE_COUNT)]
    forecasters = {
        'oracle': lambda prefix: [process.sample_suffix(prefix, rng) for _ in range(PREDICTED_DRAW_COUNT)],
        'mode': lambda prefix: [process.mode_suffix(prefix)],
    }
    return {name: evaluate(predict(forecaster, cases)) for name, forecaster in forecasters.items()}


def run(seed: np.random.SeedSequence) -> dict[str, dict[str, float]]:
    rng = np.random.default_rng(seed)
    return compare(randomize(process, rng), rng)


results = [run(seed) for seed in np.random.SeedSequence(SEED).spawn(SETTING_COUNT)]
oracle_scores = pd.DataFrame([result['oracle'] for result in results])
mode_scores = pd.DataFrame([result['mode'] for result in results])

The relative advantage of the mode model on a metric is its improvement over the oracle, divided by
the oracle's score, so it is positive when the mode model scores better. The first settings show it
per metric.

In [12]:
sign = pd.Series({'DLS': 1.0, 'Length MAE': -1.0, 'Remaining Time MAE': -1.0})
advantage = sign * (mode_scores - oracle_scores) / oracle_scores
advantage.head().round(3)

,DLS,Length MAE,Remaining Time MAE
0,0.025,0.505,0.311
1,0.022,0.480,0.359
2,0.063,0.267,0.335
3,0.000,0.394,0.304
4,0.001,0.005,0.365


For each metric, we report how often it scores the mode model at least as well as the oracle, and the
median relative advantage of the mode model.

In [13]:
pd.DataFrame(
    {
        'mode at least as good': (advantage >= 0.0).mean(),
        'median relative advantage': advantage.median(),
    }
).round(3)

,mode at least as good,median relative advantage
DLS,0.99,0.042
Length MAE,0.99,0.327
Remaining Time MAE,1.00,0.310
